In [1]:
# -------------------------------------------------------------------
# Setup project path for module imports
# -------------------------------------------------------------------
#
# This step ensures that the reserbugs package (located in the "src/"
# directory of the repository) can be imported when running this
# notebook from within the "notebooks/" folder.
#
# If the reserbugs source code is not installed as a package, we
# manually add the "src/" directory to the Python path.
#
# This allows imports such as:
#
#   from reserbugs.data import CopernicusDataRetriever
#
# to work correctly.
#
# -------------------------------------------------------------------

import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent / "src"))

In [2]:
import pandas as pd

In [3]:
# ---------------------------------------------------------------------
# Import Copernicus data retriever
# ---------------------------------------------------------------------
# CopernicusDataRetriever provides an interface to download climate data
# (e.g., temperature, precipitation, wind) from the Copernicus Climate
# Data Store for a set of geographic locations.
from reserbugs.data import CopernicusDataRetriever

In [4]:
# ---------------------------------------------------------------------
# Initialize CopernicusDataRetriever with site information
# ---------------------------------------------------------------------
# The CopernicusDataRetriever requires a dictionary describing one or
# more sites for which climate data will be retrieved.
#
# Each entry in the dictionary represents a site and must include:
#
#   - "min_year"  : first year of interest
#   - "max_year"  : last year of interest
#   - "latitude"  : site latitude (decimal degrees)
#   - "longitude" : site longitude (decimal degrees)
#
# In this example, a minimal dictionary is defined for a single site:
#
#   - "test_site" corresponds to Madrid (Spain)
#   - Only the year 2020 is specified
#
# This dictionary is used to initialize the retriever object.
# ---------------------------------------------------------------------

values_dict = {
    "test_site": {
        "min_year": 2020,
        "max_year": 2020,
        "latitude": 40.4168,   # Madrid
        "longitude": -3.7038,
    }
}

retriever = CopernicusDataRetriever(values_dict)

In [5]:
# ---------------------------------------------------------------------
# Retrieve daily abiotic (climate) data from Copernicus (ERA5)
# ---------------------------------------------------------------------
# This block retrieves daily climate data for a specific location and
# time period using the CopernicusDataRetriever interface.
#
# Parameters:
#   - years  : list of years to retrieve
#   - months : list of months (zero-padded strings, e.g. "06")
#   - days   : list of days within the selected month(s)
#   - lat    : latitude of the target location (decimal degrees)
#   - lon    : longitude of the target location (decimal degrees)
#
# In this example:
#   - Data are retrieved for Madrid (Spain)
#   - For two consecutive days: June 15–16, 2020
#
# The resulting DataFrame (df) contains daily values of selected
# abiotic variables (e.g., temperature, precipitation, wind), which
# can be used for:
#
#   - high-resolution environmental analysis
#   - aggregation to coarser temporal scales
#   - comparison with biological observations
#
# ---------------------------------------------------------------------

site = values_dict["test_site"]

df = retriever.retrieve_abiotic_data_daily(
    years=["2020"],
    months=["06"],
    days=["15", "16"],
    lat=site["latitude"],
    lon=site["longitude"],
)

2026-03-30 16:18:56,408 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-30 16:18:56,410 INFO Request ID is cb99bcfa-ef97-457c-911f-d39cf5a11125
2026-03-30 16:18:56,487 INFO status has been updated to accepted
2026-03-30 16:19:10,279 INFO status has been updated to running
2026-03-30 16:19:17,942 INFO status has been updated to successful


68adaf0d433f58e3072d2a5d255d8515.zip:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

In [6]:
# ---------------------------------------------------------------------
# Inspect retrieved dataset
# ---------------------------------------------------------------------
# Display basic information about the downloaded data, including:
#   - number of rows and columns
#   - variable names
#   - first observations
# ---------------------------------------------------------------------

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

Shape: (8, 8)

Columns:
['valid_time', 'latitude', 'longitude', 'tp', 'expver', 't2m', 'slt', 'cvl']

First rows:


,valid_time,latitude,longitude,tp,expver,t2m,slt,cvl
0,2020-06-15 00:00:00,40.5,-3.75,0.0,0001,290.583740,3.0,0.210714
1,2020-06-15 06:00:00,40.5,-3.75,0.0,0001,287.536621,3.0,0.210714
2,2020-06-15 12:00:00,40.5,-3.75,0.0,0001,298.768799,3.0,0.210714
3,2020-06-15 18:00:00,40.5,-3.75,0.0,0001,299.069824,3.0,0.210714
4,2020-06-16 00:00:00,40.5,-3.75,0.0,0001,291.603760,3.0,0.210714


In [7]:
def aggregate_daily(df):
    """
    Aggregate sub-daily climate data into daily summaries.

    This function groups observations by date and geographic location
    (latitude, longitude) and computes daily aggregated values for
    selected abiotic variables.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset containing sub-daily climate observations with
        at least the following columns:
        - 'valid_time' : timestamp of the observation (datetime-like)
        - 'latitude'   : latitude of the location
        - 'longitude'  : longitude of the location
        - 't2m'        : 2-meter air temperature
        - 'tp'         : total precipitation (accumulated per time step)
        - 'slt', 'cvl' : static or slowly varying variables

    Returns
    -------
    pandas.DataFrame
        Daily aggregated dataset where:
        - 't2m' is averaged over the day
        - 'tp' is summed over the day (total daily precipitation)
        - 'slt' and 'cvl' retain a single representative value
        - one row per (date, latitude, longitude)

    Notes
    -----
    - The 'valid_time' column is converted to calendar date
      (YYYY-MM-DD) before aggregation.
    - Precipitation ('tp') is assumed to be an accumulated variable,
      and therefore summed to obtain daily totals.
    - Static variables (e.g., soil or vegetation properties) are
      assumed constant within a day and are preserved using the
      first available value.
    """
    
    df["date"] = df["valid_time"].dt.date

    agg = (
        df.groupby(["date", "latitude", "longitude"])
        .agg({
            "t2m": "mean",   # temperature → mean
            "tp": "sum",     # precipitation → sum
            "slt": "first",  # static → keep one value
            "cvl": "first",
        })
        .reset_index()
    )

    return agg

In [9]:
# ---------------------------------------------------------------------
# Aggregate sub-daily data to daily resolution
# ---------------------------------------------------------------------
# This step converts sub-daily climate observations into daily values
# using the aggregation rules defined in the `aggregate_daily` function.
#
# The resulting dataset (df_daily) contains one row per day and location,
# with:
#   - temperature averaged over the day
#   - precipitation summed to daily totals
#   - static variables preserved
# ---------------------------------------------------------------------

df_daily = aggregate_daily(df)
df_daily.head()

,date,latitude,longitude,t2m,tp,slt,cvl
0,2020-06-15,40.5,-3.75,293.989746,0.000000,3.0,0.210714
1,2020-06-16,40.5,-3.75,293.712219,0.000002,3.0,0.210714


In [10]:
# ---------------------------------------------------------------------
# Save daily aggregated dataset
# ---------------------------------------------------------------------
# This block stores the dataset obtained after aggregating sub-daily
# climate data to daily resolution.
#
# Saving this intermediate dataset allows:
#   - skipping data retrieval and preprocessing steps
#   - ensuring reproducibility of the workflow
#   - comparing different aggregation strategies
#
# The dataset is saved in CSV format for portability and ease of use.
# ---------------------------------------------------------------------

output_path = Path("../outputs/abiotic_data/series_4_RESERBUGS_EXAMPLES_daily.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_daily.to_csv(output_path, index=False)